# 06 — Final academic aggregation and fair CPU benchmark

Attach the successful saved outputs from notebooks **01–05** as Kaggle kernel sources. This notebook checks out a pinned report-code commit, validates all raw evidence, computes eight-seed statistics, reruns the fair single-process CPU benchmark, and exports the final thesis/paper bundle.

**Kaggle settings:** Internet ON, NVIDIA Tesla T4, 12-hour timeout. The benchmark itself forces PyTorch and BLAS to one CPU thread.


In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Juliolayme/STAR_RIS_RSMA_TD3.git"
REPO_REF = "agent/td3-qos-scalability-v2"
REPORT_COMMIT = "12355fa420fedc44ba7bfe11ad7fd4f610839bcd"
REPO_DIR = Path("/kaggle/working/STAR_RIS_RSMA_TD3")

def run(command: list[str], *, cwd: Path) -> None:
    """Run one visible command and stop immediately on failure."""
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run(["git", "clone", "--filter=blob:none", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], cwd=Path("/kaggle/working"))
run(["git", "checkout", "--detach", REPORT_COMMIT], cwd=REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "-e", str(REPO_DIR), "tabulate"], cwd=REPO_DIR)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if actual_commit != REPORT_COMMIT:
    raise RuntimeError(f"Report-code drift: {actual_commit}")
run([sys.executable, "-m", "py_compile", "scripts/kaggle_final_report_common.py", "scripts/kaggle_final_report_quality.py", "scripts/kaggle_final_report_latency.py", "scripts/kaggle_final_report_main.py"], cwd=REPO_DIR)
print("Pinned report commit:", actual_commit)


## Validate all stages, compute statistics, and rerun latency


In [ ]:
run([sys.executable, "scripts/kaggle_final_report_main.py"], cwd=REPO_DIR)


## Inspect submission-ready outputs


In [ ]:
import pandas as pd

FINAL_ROOT = Path("/kaggle/working/FINAL_THESIS_PAPER_BUNDLE")
required = [
    FINAL_ROOT / "RESULTS_README.md",
    FINAL_ROOT / "REPRODUCIBILITY_MANIFEST.json",
    FINAL_ROOT / "tables" / "TABLE_FINAL_PERFORMANCE.csv",
    FINAL_ROOT / "tables" / "TABLE_CPU_LATENCY.csv",
    Path("/kaggle/working/FINAL_THESIS_PAPER_BUNDLE.zip"),
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise RuntimeError(f"Missing final deliverables: {missing}")
display(pd.read_csv(FINAL_ROOT / "tables" / "TABLE_FINAL_PERFORMANCE.csv"))
display(pd.read_csv(FINAL_ROOT / "tables" / "TABLE_CPU_LATENCY.csv"))
print((FINAL_ROOT / "RESULTS_README.md").read_text(encoding="utf-8"))
print("Final archive:", required[-1])
